# Parte 1 · Recolección de titulares de noticias peruanas

Esta es la primera de las tres partes de la tarea: armar el conjunto de datos. En la Parte 2 los titulares se etiquetarán con un LLM y se hará el fine-tune de un Transformer ligero; en la Parte 3 se desplegará un dashboard.

Se trabaja con un solo medio, **El Comercio** (opción B del enunciado), y con cuatro categorías fijas: `politica`, `economia`, `deportes` y `ciencia_tecnologia`. Lo que no encaja en ninguna se descarta.

Pasos del notebook:

1. Revisar qué permite el `robots.txt`.
2. Descargar el archivo de cuatro secciones del diario.
3. Limpiar textos y fechas.
4. Quitar duplicados.
5. Mapear la sección original de cada nota a una de las cuatro categorías.
6. Verificar que cada URL responde.
7. Controlar que el total quede entre 350 y 400 titulares.
8. Revisar el resultado con tres gráficos.
9. Guardar `datos/noticias.csv`.

## 0. Librerías y reglas de cortesía

Todas las descargas pasan por la función `pedir`, que aplica las tres reglas del enunciado:

- **User-Agent descriptivo**: se presenta como el bot de un proyecto académico, sin datos personales.
- **Pausa de 1 a 2 segundos**, elegida al azar, después de cada pedido (también después de un reintento).
- **Máximo un reintento**, y solo cuando tiene sentido: error de red, error del servidor (5xx) o aviso de exceso de pedidos (429). Un 404 no se reintenta porque volvería a fallar.

Hace falta tener instalados `requests`, `beautifulsoup4`, `pandas`, `plotly` y `nbformat`. Plotly usa `nbformat` para dibujar los gráficos dentro del notebook.

In [1]:
import json
import random
import re
import time
import xml.etree.ElementTree as ET
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup
from IPython.display import display

AGENTE = "TareaUniversitariaBot"
USER_AGENT = f"{AGENTE}/1.0 (proyecto academico de clasificacion de titulares; uso no comercial)"
BASE = "https://elcomercio.pe"
SECCIONES = ["politica", "economia", "deporte-total", "tecnologia"]
RSS = BASE + "/arc/outboundfeeds/rss/category/{seccion}/?outputType=xml"
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
PAUSA = (1, 2)
MAX_REINTENTOS = 1
RUTA_CSV = Path("datos") / "noticias.csv"

sesion = requests.Session()
sesion.headers["User-Agent"] = USER_AGENT


def pedir(url, metodo="GET"):
    respuesta = None
    for _ in range(1 + MAX_REINTENTOS):
        try:
            respuesta = sesion.request(metodo, url, timeout=20, allow_redirects=True)
        except requests.RequestException:
            respuesta = None
        time.sleep(random.uniform(*PAUSA))
        if respuesta is not None and respuesta.status_code < 500 and respuesta.status_code != 429:
            break
    return respuesta

## 1. Qué permite el robots.txt de El Comercio

Antes de pedir cualquier página se revisa `https://elcomercio.pe/robots.txt`. Tiene dos bloques de reglas:

- **Para cualquier agente** (`User-agent: *`) prohíbe solo cuatro rutas: `/catalogo/`, `/buscar/` y dos tramos del archivo general por fechas, `/archivo/todas/19*` y `/archivo/todas/200*` (los años 1900 a 2009).
- **Para 27 scrapers con nombre propio** (HTTrack, wget, libwww, WebCopier, entre otros) prohíbe todo el sitio con `Disallow: /`. Nuestro User-Agent no figura en esa lista, así que le corresponde el primer bloque.

Las rutas que usa este notebook son el archivo por sección (`/archivo/politica/`, `/archivo/economia/`, `/archivo/deporte-total/` y `/archivo/tecnologia/`) y, en el paso de verificación, la página de cada nota. Ninguna cae en las rutas prohibidas: **están permitidas**. El archivo por sección no es el archivo general `/archivo/todas/`, que es el único que tiene tramos vedados.

La celda siguiente imprime el archivo y lo comprueba en código. No se usa `urllib.robotparser` de la librería estándar porque no entiende el comodín `*`: daría por permitida `/archivo/todas/2005-01-01/`, que está prohibida. En su lugar se aplica la regla del estándar RFC 9309: gana la regla más larga que coincide con la ruta y, si dos empatan, gana `Allow`. Las dos últimas filas de la tabla son controles que deben salir prohibidos.

In [2]:
def reglas_robots(texto, agente):
    grupos, actual, leyendo_agentes = [], None, False
    for linea in texto.splitlines():
        linea = linea.split("#", 1)[0].strip()
        if ":" not in linea:
            continue
        campo, valor = (parte.strip() for parte in linea.split(":", 1))
        campo = campo.lower()
        if campo == "user-agent":
            if not leyendo_agentes:
                actual = {"agentes": [], "reglas": []}
                grupos.append(actual)
                leyendo_agentes = True
            actual["agentes"].append(valor.lower())
        elif campo in ("allow", "disallow") and actual:
            leyendo_agentes = False
            if valor:
                actual["reglas"].append((campo, valor))
    propios = [g for g in grupos if agente.lower() in g["agentes"]]
    elegidos = propios or [g for g in grupos if "*" in g["agentes"]]
    return [regla for g in elegidos for regla in g["reglas"]]


def permitido(ruta, reglas):
    coincidencias = []
    for tipo, patron in reglas:
        regex = re.escape(patron).replace(r"\*", ".*")
        if regex.endswith(r"\$"):
            regex = regex[:-2] + "$"
        if re.match(regex, ruta):
            coincidencias.append((len(patron), tipo == "allow"))
    return max(coincidencias)[1] if coincidencias else True


robots = pedir(f"{BASE}/robots.txt")
print(robots.text)

reglas = reglas_robots(robots.text, AGENTE)
rutas = [f"/archivo/{s}/" for s in SECCIONES] + ["/archivo/todas/2005-01-01/", "/buscar/peru/"]
pd.DataFrame({"ruta": rutas, "permitida": [permitido(ruta, reglas) for ruta in rutas]})

# DIRECTIVAS GENERALES
User-agent: *
Disallow: /catalogo/
Disallow: /buscar/
Disallow: /archivo/todas/19*
Disallow: /archivo/todas/200*

# BLOQUEO DE SCRAPERS ANTIGUOS Y MALICIOSOS
User-agent: UbiCrawler
User-agent: DOC
User-agent: Zao
User-agent: sitecheck.internetseer.com
User-agent: Zealbot
User-agent: MSIECrawler
User-agent: SiteSnagger
User-agent: WebStripper
User-agent: WebCopier
User-agent: Fetch
User-agent: Offline Explorer
User-agent: Teleport
User-agent: TeleportPro
User-agent: WebZIP
User-agent: linko
User-agent: HTTrack
User-agent: Microsoft.URL.Control
User-agent: Xenu
User-agent: larbin
User-agent: libwww
User-agent: ZyBORG
User-agent: Download Ninja
User-agent: wget
User-agent: grub-client
User-agent: k2spider
User-agent: NPBot
User-agent: WebReaper
Disallow: /

# SITEMAPS 
Sitemap: https://elcomercio.pe/sitemap/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/politica/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/peru/?outputType=xml
Sitemap: https://elco

,ruta,permitida
0,/archivo/politica/,True
1,/archivo/economia/,True
2,/archivo/deporte-total/,True
3,/archivo/tecnologia/,True
4,/archivo/todas/2005-01-01/,False
5,/buscar/peru/,False


## 2. Descarga del archivo por sección

El Comercio tiene una página de archivo por sección (`/archivo/politica/`, por ejemplo) que lista sus 100 notas más recientes. Con cuatro pedidos, uno por sección, se cubren las cuatro categorías.

El listado visible trae titular, bajada y enlace, pero no la fecha. La misma página incluye en su HTML un bloque JSON (`Fusion.contentCache`) con esos datos de cada nota, más la fecha de publicación y la sección que le asignó el medio. Por eso se lee ese bloque en lugar de recorrer las etiquetas `<h2>`: viene en la misma respuesta y no requiere ningún pedido adicional.

**Plan B: RSS oficial.** Si el archivo de una sección devuelve error o deja de traer el JSON, el código usa el RSS oficial de esa sección y lo avisa en pantalla. En la corrida guardada (18 de octubre de 2026) el HTML de las cuatro secciones respondió sin bloqueos y el RSS no hizo falta: la columna `fuente` de la tabla de abajo dice `archivo HTML` en las cuatro. Los RSS traen menos notas que el archivo, así que si se activara es probable que no se llegue a 350 titulares.

In [3]:
def leer_archivo(html):
    inicio = html.index("Fusion.contentCache=") + len("Fusion.contentCache=")
    cache, _ = json.JSONDecoder().raw_decode(html, inicio)
    consultas = cache["story-feed-by-section"].values()
    notas = max(consultas, key=lambda c: len(c["data"]["content_elements"]))["data"]["content_elements"]
    filas = []
    for nota in notas:
        web = (nota.get("websites") or {}).get("elcomercio") or {}
        if not web.get("website_url"):
            continue
        filas.append({
            "titular": (nota.get("headlines") or {}).get("basic"),
            "bajada": (nota.get("subheadlines") or {}).get("basic"),
            "url": urljoin(BASE, web["website_url"]),
            "seccion_original": (web.get("website_section") or {}).get("path", "").strip("/"),
            "fecha": nota.get("display_date"),
        })
    return filas


def leer_rss(xml):
    filas = []
    for item in ET.fromstring(xml).iter("item"):
        url = item.findtext("link", "").strip()
        filas.append({
            "titular": item.findtext("title"),
            "bajada": BeautifulSoup(item.findtext("description", ""), "html.parser").get_text(" ", strip=True),
            "url": url,
            "seccion_original": "/".join(urlparse(url).path.strip("/").split("/")[:-1]),
            "fecha": parsedate_to_datetime(item.findtext("pubDate")).isoformat(),
        })
    return filas


filas, fuente = [], {}
for seccion in SECCIONES:
    respuesta = pedir(f"{BASE}/archivo/{seccion}/")
    try:
        if respuesta is None or respuesta.status_code != 200:
            raise ValueError(f"HTTP {getattr(respuesta, 'status_code', 'sin respuesta')}")
        nuevas, fuente[seccion] = leer_archivo(respuesta.text), "archivo HTML"
    except (ValueError, KeyError, TypeError) as error:
        print(f"{seccion}: el archivo HTML no sirvió ({error}); se usa el RSS oficial")
        nuevas, fuente[seccion] = leer_rss(pedir(RSS.format(seccion=seccion)).content), "RSS"
    filas += [{**fila, "seccion_archivo": seccion} for fila in nuevas]

df_crudo = pd.DataFrame(filas)
print(f"Titulares descargados: {len(df_crudo)}")
pd.DataFrame({"fuente": fuente, "titulares": df_crudo["seccion_archivo"].value_counts()})

Titulares descargados: 400


,fuente,titulares
politica,archivo HTML,100
economia,archivo HTML,100
deporte-total,archivo HTML,100
tecnologia,archivo HTML,100


## 3. Limpieza de textos y fechas

Dos arreglos antes de seguir:

- **Espacios.** Algunos titulares traen espacios dobles o sobrantes al inicio o al final. Se normalizan para que la deduplicación no trate como distintos dos textos iguales.
- **Fechas.** El JSON da la fecha en UTC y no siempre con el mismo formato: unas traen milisegundos y otras no. Por eso se lee con `format="ISO8601"`. Después se pasa a hora de Lima y se guarda solo el día, porque una nota publicada a las 10 p. m. del 16 en Lima figura en UTC como del 17.

La tabla muestra qué período cubre el archivo de cada sección.

In [4]:
df = df_crudo.copy()
for columna in ["titular", "bajada"]:
    df[columna] = df[columna].fillna("").str.split().str.join(" ")
df = df[df["titular"] != ""]
df["fecha"] = pd.to_datetime(df["fecha"], utc=True, format="ISO8601").dt.tz_convert("America/Lima").dt.date
df["medio"] = "El Comercio"

print(f"Titulares sin bajada: {(df['bajada'] == '').sum()}")
print(f"Fechas: del {df['fecha'].min()} al {df['fecha'].max()}")
df.groupby("seccion_archivo")["fecha"].agg(desde="min", hasta="max")

Titulares sin bajada: 0
Fechas: del 2026-10-06 al 2026-10-18


,desde,hasta
seccion_archivo,,
deporte-total,2026-10-14,2026-10-18
economia,2026-10-12,2026-10-17
politica,2026-10-12,2026-10-18
tecnologia,2026-10-06,2026-10-17


## 4. Deduplicación

Una misma nota puede figurar en el archivo de dos secciones. En la corrida guardada pasó una vez: una nota de `tecnologia/ciencias` sobre una flor descubierta en los Andes y bautizada en honor a Messi apareció en el archivo de deportes y en el de tecnología.

Los duplicados se quitan en dos pasadas: primero por URL y después por titular, sin distinguir mayúsculas, por si el medio publicó el mismo titular en dos direcciones distintas. La tabla de abajo muestra las filas repetidas por URL.

In [5]:
antes = len(df)
repetidas = df[df["url"].duplicated(keep=False)]
df = df.drop_duplicates(subset="url")
sin_repetir_url = len(df)
df = df[~df["titular"].str.casefold().duplicated()]

print(f"Duplicados por URL: {antes - sin_repetir_url}")
print(f"Duplicados por titular: {sin_repetir_url - len(df)}")
print(f"Total de duplicados quitados: {antes - len(df)}. Quedan {len(df)} titulares.")
repetidas[["seccion_archivo", "seccion_original", "titular"]]

Duplicados por URL: 1
Duplicados por titular: 0
Total de duplicados quitados: 1. Quedan 399 titulares.


,seccion_archivo,seccion_original,titular
243,deporte-total,tecnologia/ciencias,Descubren en los Andes peruanos una nueva flor...
304,tecnologia,tecnologia/ciencias,Descubren en los Andes peruanos una nueva flor...


## 5. Mapeo de la sección original a la categoría

`seccion_original` es la ruta de la sección que el propio medio asignó a cada nota, por ejemplo `economia/mercados`. La categoría se decide por el primer tramo de esa ruta:

| Primer tramo de `seccion_original` | Categoría |
|---|---|
| `politica`, `elecciones` | `politica` |
| `economia` | `economia` |
| `deporte-total` | `deportes` |
| `tecnologia` (incluye `tecnologia/ciencias` y `tecnologia/ecologia`) | `ciencia_tecnologia` |
| cualquier otro (`opinion`, `ecdata`, `bbc-news-mundo`, `ruedas-tuercas`…) | se descarta |

No hay una quinta categoría: lo que no encaja se descarta y se cuenta. Dos efectos del criterio:

- Las columnas de `economia/opinion` quedan en `economia`, mientras que las de `opinion/columnistas` se descartan aunque traten de política. El criterio es la sección, no el tema.
- Por lo mismo, la etiqueta dice dónde publicó el medio la nota, no qué dice el titular.

La tabla muestra el mapeo de cada sección que apareció en los datos y cuántos titulares tiene.

In [6]:
MAPEO = {"politica": "politica", "elecciones": "politica", "economia": "economia",
         "deporte-total": "deportes", "tecnologia": "ciencia_tecnologia"}
df["etiqueta"] = df["seccion_original"].str.split("/").str[0].map(MAPEO)

tabla_mapeo = (df.groupby("seccion_original", as_index=False)
                 .agg(categoria=("etiqueta", "first"), titulares=("url", "size"))
                 .fillna({"categoria": "(descartada)"}))
tabla_mapeo["categoria"] = pd.Categorical(tabla_mapeo["categoria"], CATEGORIAS + ["(descartada)"])
display(tabla_mapeo.sort_values(["categoria", "titulares"], ascending=[True, False], ignore_index=True))

descartadas = df["etiqueta"].isna()
print(f"Descartadas por no encajar en ninguna de las cuatro categorías: {descartadas.sum()}")
print(df.loc[descartadas, "seccion_original"].value_counts().to_string())
df = df[~descartadas]
print(f"Quedan {len(df)} titulares")

,seccion_original,categoria,titulares
0,politica/congreso,politica,26
1,politica,politica,25
2,politica/gobierno,politica,18
3,politica/actualidad,politica,8
4,politica/internacional,politica,5
5,politica/elecciones,politica,4
6,politica/justicia,politica,3
7,elecciones,politica,2
8,politica/partidos,politica,1
9,economia/peru,economia,46


Descartadas por no encajar en ninguna de las cuatro categorías: 15
seccion_original
opinion/columnistas          6
ecdata                       4
ruedas-tuercas/automotriz    2
bbc-news-mundo               2
eldominical/ciencia          1
Quedan 384 titulares


## 6. Verificación de URLs

Se comprueba que cada URL responde con un pedido `HEAD`, que solo trae las cabeceras y no descarga la página. Antes de cada pedido se confirma con las reglas del `robots.txt` que la ruta está permitida. Las URLs que no devuelven 200 se muestran y se quitan.

Con la pausa obligatoria de 1 a 2 segundos, revisar unas 380 URLs toma alrededor de 10 minutos.

In [7]:
print(f"Verificando {len(df)} URLs (unos {len(df) * 1.5 / 60:.0f} minutos)")
estados = []
for i, url in enumerate(df["url"], start=1):
    if not permitido(urlparse(url).path, reglas):
        estados.append("prohibida por robots.txt")
        continue
    respuesta = pedir(url, metodo="HEAD")
    estados.append(respuesta.status_code if respuesta is not None else "sin respuesta")
    if i % 50 == 0:
        print(f"  {i}/{len(df)}")

df["estado_http"] = estados
print(df["estado_http"].value_counts().to_string())
caidas = df[df["estado_http"] != 200]
print(f"URLs que no respondieron 200: {len(caidas)}")
display(caidas[["titular", "url", "estado_http"]])
df = df[df["estado_http"] == 200]

Verificando 384 URLs (unos 10 minutos)
  50/384
  100/384
  150/384
  200/384
  250/384
  300/384
  350/384
estado_http
200    384
URLs que no respondieron 200: 0


,titular,url,estado_http


## 7. Control de volumen

El enunciado pide entre 350 y 400 titulares. Aquí se arma la tabla final con las siete columnas pedidas, en este orden, y se comprueba el rango.

In [8]:
COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
noticias = df[COLUMNAS].reset_index(drop=True)

cumple = 350 <= len(noticias) <= 400
print(f"Titulares finales: {len(noticias)} (rango pedido 350-400: {'cumple' if cumple else 'NO cumple'})")
print(noticias["etiqueta"].value_counts().reindex(CATEGORIAS).to_string())
noticias.head()

Titulares finales: 384 (rango pedido 350-400: cumple)
etiqueta
politica              92
economia              99
deportes              98
ciencia_tecnologia    95


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Diputados: Informe recomienda suspender por 12...,El documento será debatido este lunes por los ...,El Comercio,politica,https://elcomercio.pe/politica/diputados-infor...,2026-10-18,politica
1,Comisión de Constitución del Senado reanuda es...,El grupo de trabajo que encabeza el senador Ca...,El Comercio,politica,https://elcomercio.pe/politica/comision-de-con...,2026-10-18,politica
2,Facultades legislativas: Senado sigue sin alca...,SITUACIÓN. El presidente de la Comisión de Con...,El Comercio,politica,https://elcomercio.pe/politica/facultades-legi...,2026-10-17,politica
3,Renovación Popular se declara en reorganizació...,La medida deja sin efecto las credenciales de ...,El Comercio,politica,https://elcomercio.pe/politica/renovacion-popu...,2026-10-17,politica
4,"Alfonso, ¿ahora qué?, una crónica de Fernando ...",El líder de Ahora Nación y portavoz de su banc...,El Comercio,politica,https://elcomercio.pe/politica/alfonso-ahora-q...,2026-10-17,politica


## 8. Gráficos

Tres gráficos con Plotly Express. El título de cada uno enuncia la conclusión y se calcula con los datos, así que se actualiza si se repite la recolección; el subtítulo dice qué se mide. Los tres usan un solo color porque cada barra o caja ya está nombrada en su eje y el color no tiene nada que distinguir.

La función `mostrar` aplica el mismo estilo a los tres: fondo claro, cuadrícula tenue y textos en gris.

In [9]:
AZUL = "#2a78d6"
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]


def rango_es(inicio, fin):
    mismo_mes = (inicio.year, inicio.month) == (fin.year, fin.month)
    desde = str(inicio.day) if mismo_mes else f"{inicio.day} de {MESES[inicio.month - 1]}"
    return f"del {desde} al {fin.day} de {MESES[fin.month - 1]}"


def mostrar(fig, subtitulo):
    fig.update_layout(
        template="simple_white", paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
        font=dict(family="system-ui, 'Segoe UI', sans-serif", size=13, color="#0b0b0b"),
        title=dict(font_size=17, subtitle=dict(text=subtitulo, font=dict(size=13, color="#52514e"))),
        margin=dict(t=115, r=90), barcornerradius=4, bargap=0.35,
    )
    fig.update_xaxes(linecolor="#c3c2b7", ticks="", tickfont_color="#52514e", title_font_color="#52514e")
    fig.update_yaxes(linecolor="#c3c2b7", ticks="", showgrid=True, gridcolor="#e1e0d9",
                     tickfont_color="#52514e", title_font_color="#52514e")
    fig.show()

### Gráfico 1 · ¿Las categorías quedan balanceadas?

Para el fine-tune de la Parte 2 conviene que ninguna categoría domine: si una tuviera el doble de ejemplos que otra, el modelo tendería a predecirla de más.

In [10]:
conteo = noticias["etiqueta"].value_counts().reindex(CATEGORIAS)
if conteo.max() <= 1.25 * conteo.min():
    titulo = f"Las cuatro categorías quedan balanceadas: entre {conteo.min()} y {conteo.max()} titulares cada una"
else:
    titulo = f"{conteo.idxmax()} tiene {conteo.max() / conteo.min():.1f} veces más titulares que {conteo.idxmin()}"

fig = px.bar(conteo.rename_axis("etiqueta").reset_index(name="titulares"), x="etiqueta", y="titulares",
             text="titulares", title=titulo, color_discrete_sequence=[AZUL],
             labels={"etiqueta": "Categoría", "titulares": "Titulares"})
fig.update_traces(textposition="outside", cliponaxis=False, hovertemplate="%{x}: %{y} titulares<extra></extra>")
mostrar(fig, "Titulares por categoría en el dataset final")

### Gráfico 2 · ¿Qué tan largos son los titulares?

El largo sirve para fijar la longitud máxima de entrada del Transformer en la Parte 2. Aquí se cuenta en palabras; el modelo trabaja con tokens, que serán algo más numerosos.

In [11]:
largo = noticias.assign(palabras=noticias["titular"].str.split().str.len())
p95 = int(largo["palabras"].quantile(0.95))
mas_largos = largo.groupby("etiqueta")["palabras"].mean().idxmax()

fig = px.box(largo, x="etiqueta", y="palabras", points="all", hover_data=["titular"],
             category_orders={"etiqueta": CATEGORIAS}, color_discrete_sequence=[AZUL],
             title=f"El 95% de los titulares tiene {p95} palabras o menos; los de {mas_largos} son los más largos",
             labels={"etiqueta": "Categoría", "palabras": "Palabras por titular"})
fig.update_traces(jitter=0.5, pointpos=0, marker=dict(size=8, opacity=0.35, line=dict(width=1, color="#fcfcfb")))
fig.add_hline(y=p95, line_width=1, line_color="#898781",
              annotation_text=f"95% ≤ {p95}", annotation_position="right", annotation_font_color="#52514e")
mostrar(fig, "Cada punto es un titular; la caja abarca la mitad central y su línea interior es la mediana")

### Gráfico 3 · ¿Qué período cubren los datos?

El archivo entrega las 100 notas más recientes de cada sección, así que cada categoría cubre un tramo de días distinto según cuánto publica su sección. Debajo del gráfico va la tabla con los conteos por día.

In [12]:
por_dia = noticias.groupby(["etiqueta", "fecha"]).size().reset_index(name="titulares")
inicio, fin = noticias["fecha"].min(), noticias["fecha"].max()
tramo = noticias.groupby("etiqueta")["fecha"].agg(lambda f: (f.max() - f.min()).days + 1)
corto = tramo.idxmin()

fig = px.bar(por_dia, x="fecha", y="titulares", facet_col="etiqueta", facet_col_wrap=1, facet_row_spacing=0.07,
             category_orders={"etiqueta": CATEGORIAS}, color_discrete_sequence=[AZUL], height=640,
             title=f"Todo el dataset cabe en {(fin - inicio).days + 1} días, y {corto} solo abarca {tramo[corto]} de ellos",
             labels={"fecha": "Día de publicación", "titulares": "Titulares"})
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], x=0, xanchor="left", font_color="#0b0b0b"))
fig.update_xaxes(dtick=86400000, tickformat="%d/%m")
fig.update_yaxes(title_text="")
mostrar(fig, f"Titulares publicados por día, {rango_es(inicio, fin)} (hora de Lima)")
pd.crosstab(noticias["fecha"], noticias["etiqueta"]).reindex(columns=CATEGORIAS)

etiqueta,politica,economia,deportes,ciencia_tecnologia
fecha,,,,
2026-10-06,0,0,0,14
2026-10-07,0,0,0,6
2026-10-08,0,0,0,11
2026-10-09,0,0,0,5
2026-10-10,0,0,0,6
2026-10-11,0,0,0,7
2026-10-12,20,8,0,10
2026-10-13,15,14,0,5
2026-10-14,20,17,5,8


## 9. Guardar el CSV

Se guarda en `datos/noticias.csv`, una ruta relativa a la carpeta del notebook (`tarea_1/`), con `index=False` y `encoding="utf-8-sig"` para que Excel lea bien las tildes. Después se vuelve a leer el archivo para comprobar filas y columnas.

In [13]:
RUTA_CSV.parent.mkdir(exist_ok=True)
noticias.to_csv(RUTA_CSV, index=False, encoding="utf-8-sig")

leido = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")
print(f"Guardado {RUTA_CSV}: {leido.shape[0]} filas")
print(f"Columnas: {list(leido.columns)}")

Guardado datos\noticias.csv: 384 filas
Columnas: ['titular', 'bajada', 'medio', 'seccion_original', 'url', 'fecha', 'etiqueta']


## Limitaciones

- **Ventana de tiempo corta.** El archivo de cada sección solo entrega sus 100 notas más recientes, así que todos los titulares son de unos pocos días. En la corrida que generó `datos/noticias.csv` (18 de octubre de 2026, 384 titulares) las fechas van del 6 al 18 de octubre, 13 días en total, y el tramo cambia según la categoría: deportes cubre solo 5 días (del 14 al 18), economía 6 (del 12 al 17), política 7 (del 12 al 18) y ciencia y tecnología 12 (del 6 al 17). Los titulares reflejan la coyuntura de esos días: 16 tratan del debate de las facultades legislativas en el Senado y 15 del Torneo Clausura, entre la Liga 1 y la liga argentina. Un modelo entrenado con ellos puede aprender esos temas puntuales en vez de rasgos generales de cada categoría. Si se repite la recolección otro día, el rango cambia; el de cada corrida aparece en el paso 3 y en el gráfico 3.
- **Un solo medio.** Todo viene de El Comercio, con su estilo de redacción y su forma de organizar las secciones.
- **Etiqueta por sección.** La etiqueta dice dónde publicó el medio la nota, no de qué trata el titular, así que habrá casos dudosos, como una nota de economía sobre una ley del Congreso.
- **Descartes.** Las notas de opinión general, de ecdata y de otras secciones ajenas a las cuatro categorías se pierden, aunque varias traten esos temas.